# Example 6.4.2 / 1: A Compatible Redeclaration

Companion notebook for **6.4.2 Deleting and Redeclaring Components** in the AMPL lab Documentation, section "A Compatible Redeclaration". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## A Compatible Redeclaration

Because `capacity` stays a scalar parameter, the constraint that uses it remains valid and picks
up the new default at the next solve. amplpy has no method for structural commands, so the Python
tab sends it with `ampl.eval()`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param capacity >= 0 default 100;     # production capacity, units
var Make >= 0;                       # units made
maximize Output: Make;               # make as much as possible
subject to Limit: Make <= capacity;  # production cannot exceed capacity

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
ampl.option["solver_msg"] = 0
ampl.eval("redeclare param capacity >= 0 default 120;  # still a scalar; only the default changes")
ampl.solve(solver="highs")
print(ampl.var["Make"].value())

120.0


### Pure AMPL: model.run

The same in pure AMPL, with the same `model.mod`:

In [5]:
%%ampl_eval
reset;
reset options;
model model.mod;

In [6]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
redeclare param capacity >= 0 default 120;  # still a scalar; only the default changes
solve;
display Make;

Make = 120

